In [1]:
import torch
from datasets import load_dataset
from transformers import (
    BitsAndBytesConfig,
    AutoModelForCausalLM,
    AutoTokenizer
)
from huggingface_hub import snapshot_download
from peft import PeftModel, LoraConfig
from trl import SFTTrainer, SFTConfig

d:\python-workspace\NL2SQL-chat\myenv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# snapshot_download(
#     repo_id="Qwen/Qwen2.5-Coder-3B-Instruct",
#     repo_type="model"
# )

In [2]:
fp16 = False
bf16 = False
compute_dtype = None
if torch.cuda.is_available():
    major, _ = torch.cuda.get_device_capability()
    if major >= 8:
        print("=== Using bf16 data type ===")
        bf16 = True
        compute_dtype = torch.bfloat16
    else:
        fp16 = True
        compute_dtype = torch.float16
else:
    compute_dtype = torch.float32

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=False
)

peft_config = LoraConfig(
    task_type="CAUSAL_LM",
    lora_alpha=8,
    lora_dropout=0.1,
    r=4,
    bias="none"
)

=== Using bf16 data type ===


In [4]:
dataset = load_dataset("b-mc2/sql-create-context")
print("Tải dataset thành công")
dataset

Tải dataset thành công


DatasetDict({
    train: Dataset({
        features: ['answer', 'question', 'context'],
        num_rows: 78577
    })
})

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_name = "Qwen/Qwen2.5-Coder-3B-Instruct"
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config = bnb_config,
    device_map=device,
    dtype=compute_dtype
)
# model.config.use_cache = False
# model.config.pretraining_tp = 1

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-Coder-3B-Instruct")

Loading weights: 100%|██████████| 434/434 [00:18<00:00, 22.92it/s] 


In [6]:
def format_data(example, tokenizer):
    messages = [
        {
            "role": "system",
            "content": f"You are an AI assistant can code SQL perfectly. Write query accurately based on this schema:\n{example['context']} "
        },
        {
            "role": "user",
            "content": example['question']
        },
        {
            "role":"assistant",
            "content": example['answer']
        }
    ]
    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False
    )
    return {"text": formatted_prompt}

dataset = dataset['train'].map(
    format_data,
    num_proc=8,
    fn_kwargs={"tokenizer":tokenizer}
)

In [7]:
dataset.remove_columns(['answer','question','context'])
dataset['text'][0]

'<|im_start|>system\nYou are an AI assistant can code SQL perfectly. Write query accurately based on this schema:\nCREATE TABLE head (age INTEGER) <|im_end|>\n<|im_start|>user\nHow many heads of the departments are older than 56 ?<|im_end|>\n<|im_start|>assistant\nSELECT COUNT(*) FROM head WHERE age > 56<|im_end|>\n'

## CẤM CHẠY CELL DƯỚI

In [8]:
train_args = SFTConfig(
    output_dir="./save_model",
    num_train_epochs=2,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    optim="paged_adamw_8bit",
    weight_decay=0.001,
    per_device_train_batch_size=4,
    logging_steps=100,
    fp16=fp16,
    bf16=bf16,
    dataset_text_field='text',
    packing=False,
    save_steps=1000,
    save_total_limit=5
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset,
    args=train_args,
    peft_config=peft_config
)
print("=== BẮT ĐẦU HUẤN LUYỆN ===")
trainer.train()
print("=== HUẤN LUYỆN THÀNH CÔNG ===")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


=== BẮT ĐẦU HUẤN LUYỆN ===


Step,Training Loss
100,1.881793
200,0.818542
300,0.771251
400,0.750356
500,0.727014
600,0.717898
700,0.698141
800,0.702149
900,0.707014
1000,0.708861


=== HUẤN LUYỆN THÀNH CÔNG ===


In [ ]:
adapter_path = r"..\save_model\checkpoint-39290"
finetuned_model = PeftModel.from_pretrained(model, adapter_path)
model.eval()
print("✅ Sẵn sàng! Mô hình đã được tải thành công.")

✅ Sẵn sàng! Mô hình đã được tải thành công.


In [5]:
def generate_input(user_prompt, schema_db):
    messages = [
        {
            "role": "system",
            "content": f"You are an AI assistant can code SQL perfectly. Write query accurately based on this schema:\n{schema_db} "
        },
        {
            "role": "user",
            "content": user_prompt
        }
    ]
    input_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    return input_text

context_5 = """CREATE TABLE students (student_id INT PRIMARY KEY, name VARCHAR(100), major VARCHAR(50));
CREATE TABLE courses (course_id INT PRIMARY KEY, course_name VARCHAR(100), credits INT);
CREATE TABLE professors (prof_id INT PRIMARY KEY, name VARCHAR(100), department VARCHAR(50));
CREATE TABLE course_assignments (course_id INT, prof_id INT, semester VARCHAR(20));
CREATE TABLE enrollments (enrollment_id INT PRIMARY KEY, student_id INT, course_id INT, grade VARCHAR(2));"""
user_prompt_5 = "Get students name and major by fixing this query: SELECT * FROM students"

input_query = generate_input(user_prompt_5, context_5)
input_query

'<|im_start|>system\nYou are an AI assistant can code SQL perfectly. Write query accurately based on this schema:\nCREATE TABLE students (student_id INT PRIMARY KEY, name VARCHAR(100), major VARCHAR(50));\nCREATE TABLE courses (course_id INT PRIMARY KEY, course_name VARCHAR(100), credits INT);\nCREATE TABLE professors (prof_id INT PRIMARY KEY, name VARCHAR(100), department VARCHAR(50));\nCREATE TABLE course_assignments (course_id INT, prof_id INT, semester VARCHAR(20));\nCREATE TABLE enrollments (enrollment_id INT PRIMARY KEY, student_id INT, course_id INT, grade VARCHAR(2)); <|im_end|>\n<|im_start|>user\nGet students name and major by fixing this query: SELECT * FROM students<|im_end|>\n<|im_start|>assistant\n'

In [ ]:
with torch.no_grad():
    tokens = tokenizer(input_query, return_tensors="pt").to(device)
    outputs = model.generate(
        **tokens,
        max_new_tokens=256,
        temperature=0.1,
        pad_token_id = tokenizer.eos_token_id
    )
    outputs_text = tokenizer.decode(
        outputs[0],
        skip_special_tokens=False,
    )
    print(outputs)
    print(outputs_text.split("assistant")[-1][:-10])

tensor([[151644,   8948,    198,   2610,    525,    458,  15235,  17847,    646,
           2038,   7870,  13942,     13,   9645,   3239,  29257,   3118,    389,
            419,  10802,    510,  22599,  14363,   4143,    320,  12038,    842,
           9221,  37467,  12013,     11,    829,  37589,      7,     16,     15,
             15,    701,   3598,  37589,      7,     20,     15,   1106,  22599,
          14363,  13980,    320,  11856,    842,   9221,  37467,  12013,     11,
           3308,   1269,  37589,      7,     16,     15,     15,    701,  20141,
           9221,    317,  22599,  14363,  44624,    320,  21826,    842,   9221,
          37467,  12013,     11,    829,  37589,      7,     16,     15,     15,
            701,   9292,  37589,      7,     20,     15,   1106,  22599,  14363,
           3308,  20688,   1368,    320,  11856,    842,   9221,     11,   2778,
            842,   9221,     11,  33153,  37589,      7,     17,     15,   1106,
          22599,  14363,  51